# transistor：復元角度と学習角度への偏りを調べる

**問い：固定回転モデルは、入力に最も近い36度刻みの角度へ寄せて復元しているのか？**

11月18日の保存済み復元画像を使い、通常／固定回転／ランダム回転の3モデルを比較します。
対象は正常テスト60枚。再学習・再推論・AUROCの再計算はしません。

1. 正しい角度が分かる入力画像と、それをぼかした画像で、角度推定方法を確認する。
2. 復元画像の見た目に最も近い角度を1度刻みの参照画像から推定する。
3. 入力の向きへの追従、最寄り学習角度への偏り、別の向きへの崩れを区別する。
4. 画像全体と中央領域、相関曲線・候補画像を見比べ、推定の曖昧さも調べる。

**コード作成・静的確認のみ。アシスタントはコード・構文テスト・描画を実行していません。**
実行はユーザーが行います。モデル重み・torch・torchvisionは不要です。

この推定は、背景も含む画像の照合です。本体だけの厳密な姿勢推定ではありません。
「近い学習角度の画像に似ている」という観察を調べるもので、モデル内部の角度選択機構の証明ではありません。


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.metadata
import json
import math
import platform
import uuid
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents)
    if (p/'memory/memory.md').is_file() and (p/'data/transistor').is_dir()),None)
if ROOT is None:raise FileNotFoundError('リポジトリ内を作業ディレクトリにしてください。')
SOURCE_RUN=ROOT/'saved_models/transistor_rotation_followup_20261118/20261118_180211_7818e304'
EVAL_DIR=SOURCE_RUN/'03_new_model_evaluation'
UNSEEN_RUN=ROOT/'saved_models/transistor_unseen_rotation_20261117/20261117_232200_a212593a'
OUTPUT_BASE=ROOT/'saved_models/transistor_rotation_angle_bias_20261119'
CONDITIONS=('A_normal','B_fixed','C_random')
LABELS={'A_normal':'通常学習','B_fixed':'固定回転','C_random':'ランダム回転'}
COLORS={'A_normal':'#526b8e','B_fixed':'darkorange','C_random':'#388b62'}
SCENARIOS={'unrotated':'回転なし','known_rotated':'既知角度',
    'midpoint':'中間角度','unseen_random':'未知ランダム角度'}
ASSIGNMENT_SEEDS=(42,43,44,45,46)
TRAINING_ANGLES=np.arange(0,360,36,dtype=int)
CANDIDATE_ANGLES=np.arange(360,dtype=int)  # 角度推定の分解能は1度

# 全60枚が初期値。少数で確認する場合は、例：NORMAL_PATHS=['good/000.png']。
NORMAL_PATHS=None
# 128×128に対する中央領域。領域間で位置合わせ方法は変えない。
ROI_BOXES={'whole':(0,0,128,128),'central':(32,32,96,96)}
ROI_LABELS={'whole':'画像全体','central':'中央64×64'}
FEATURE_SIZE=64  # 各領域を64×64へそろえ、相関計算量を抑える
ANALYSIS_BLUR_SIGMA=1.0  # 比較前に参照と対象の両方へ同じぼかし（128画像の画素単位）
CALIBRATION_BLURS=(0.0,1.0,2.0)  # 入力だけへ追加するぼかし。0は元の入力
ANGLE_TOLERANCE=3.0  # 角度集計用の説明的な許容幅。検証済みの統計的閾値ではない
OTHER_PEAK_EXCLUSION=10.0  # 最良角度の±10度を除いた別候補との相関差を見る
SMALL_PEAK_GAP=0.02  # 曖昧さの確認用の目安。確率・信頼区間ではない
LOW_CORRELATION=0.30
RETAIN_CORRELATION_CURVES=True
VISUAL_PATHS=None  # Noneなら選択した正常画像の先頭1枚。例：['good/000.png']。表示のみの設定
VISUAL_CASES=[('midpoint',42),('unseen_random',42)]
VISUAL_ROIS=('whole','central')

assert FEATURE_SIZE>0 and 0<ANGLE_TOLERANCE<9
assert 0<OTHER_PEAK_EXCLUSION<180 and ANALYSIS_BLUR_SIGMA>=0
assert all(v>=0 for v in CALIBRATION_BLURS)
assert len(set(CALIBRATION_BLURS))==len(CALIBRATION_BLURS)
assert 0.0 in CALIBRATION_BLURS
available_fonts={f.name for f in font_manager.fontManager.ttflist}
for font in ['Yu Gothic','Meiryo','MS Gothic','Noto Sans CJK JP','IPAexGothic']:
    if font in available_fonts:
        plt.rcParams['font.family']=font;break
else:
    import japanize_matplotlib
plt.rcParams.update({'font.size':11,'axes.unicode_minus':False,'figure.dpi':100})

def read_json(path):return json.loads(path.read_text(encoding='utf-8'))
def write_json(path,value):path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
def sha256(path):return hashlib.sha256(path.read_bytes()).hexdigest()
def array_hash(value):return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()
def new_id():return datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]


## 1. 保存結果と正常画像の照合

最新結果を自動選択せず、比較対象の実行IDを固定します。
参照ファイルのSHA-256、画像順、角度、元画像、入力テンソルを照合します。
12ケースを同じ正常60枚で比べます。5回は角度割り当ての違いであり、独立した300枚ではありません。
保存済み入力・復元を使用し、モデルは読み込みません。


In [ ]:
source_config=read_json(SOURCE_RUN/'session_config.json')
evaluation_config=read_json(EVAL_DIR/'config.json')
source_output_hashes=read_json(SOURCE_RUN/'output_file_hashes.json')
unseen_output_hashes=read_json(UNSEEN_RUN/'output_file_hashes.json')
manifest=read_json(SOURCE_RUN/'test_manifest.json')
assert source_config['status']==evaluation_config['status']=='completed'
assert read_json(SOURCE_RUN/'01_preprocessing_check/status.json')['status']=='passed'
assert evaluation_config['test_source_count']==len(manifest)==100
assert evaluation_config['unseen_source']==UNSEEN_RUN.relative_to(ROOT).as_posix()
assert set(evaluation_config['model_hashes'])==set(CONDITIONS)
assert manifest==read_json(UNSEEN_RUN/'test_manifest.json')
assert hashlib.sha256(json.dumps(manifest,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()==evaluation_config['test_manifest_sha256']
all_paths=[r['relative_path'] for r in manifest]
assert len(set(all_paths))==100
normal_indices=[i for i,r in enumerate(manifest) if r['category']=='good' and r['true_label']==0]
assert len(normal_indices)==60
if NORMAL_PATHS is not None:
    assert NORMAL_PATHS and len(set(NORMAL_PATHS))==len(NORMAL_PATHS)
    assert set(NORMAL_PATHS).issubset({manifest[i]['relative_path'] for i in normal_indices})
    normal_indices=[i for i in normal_indices if manifest[i]['relative_path'] in NORMAL_PATHS]
normal_records=[manifest[i] for i in normal_indices]
normal_paths=[r['relative_path'] for r in normal_records]
CASES=[('unrotated',-1),('known_rotated',42)]+[(s,seed)
    for s in ['midpoint','unseen_random'] for seed in ASSIGNMENT_SEEDS]
assert [list(c) for c in CASES]==evaluation_config['test_cases']
source_hashes={}
def register_source(path,expected=None):
    value=sha256(path)
    if expected is not None:assert value==expected,f'保存ファイルのハッシュ不一致：{path}'
    source_hashes[path.relative_to(ROOT).as_posix()]=value
    return path
for p in [SOURCE_RUN/'session_config.json',SOURCE_RUN/'test_manifest.json',
          SOURCE_RUN/'output_file_hashes.json',UNSEEN_RUN/'output_file_hashes.json',
          UNSEEN_RUN/'config.json',UNSEEN_RUN/'test_manifest.json']:
    register_source(p)
register_source(EVAL_DIR/'config.json',source_output_hashes['03_new_model_evaluation/config.json'])
for filename in ['angle_assignments.csv','rotated_input_manifest.json']:
    register_source(UNSEEN_RUN/filename,unseen_output_hashes[filename])
assignments=pd.read_csv(UNSEEN_RUN/'angle_assignments.csv')
input_records={(r['scenario'],r['assignment_seed'],r['relative_path']):r
    for r in read_json(UNSEEN_RUN/'rotated_input_manifest.json')}
originals={}
for record in normal_records:
    p=register_source(ROOT/record['repo_path'],record['file_sha256'])
    with Image.open(BytesIO(p.read_bytes())) as im:
        assert im.size==(1024,1024)
        originals[record['relative_path']]=im.convert('RGB').copy()

case_data={}
for scenario,seed in CASES:
    if scenario in ['unrotated','known_rotated']:
        p=register_source(SOURCE_RUN/'01_preprocessing_check/inputs.npz',source_output_hashes['01_preprocessing_check/inputs.npz'])
        with np.load(p,allow_pickle=False) as data:
            assert data['relative_paths'].tolist()==all_paths
            inputs=data[scenario][normal_indices,0].copy()
            angles=np.zeros(len(normal_indices)) if scenario=='unrotated' else data['known_angles'][normal_indices].copy()
    else:
        rel=f'cases/{scenario}_seed{seed}/evaluation_arrays.npz'
        p=register_source(UNSEEN_RUN/rel,unseen_output_hashes[rel])
        with np.load(p,allow_pickle=False) as data:
            assert data['relative_paths'].tolist()==all_paths
            inputs=data['rotated_inputs'][normal_indices,0].copy();angles=data['angles'][normal_indices].copy()
        part=assignments[(assignments.scenario==scenario)&(assignments.assignment_seed==seed)].set_index('relative_path').loc[normal_paths]
        assert np.array_equal(angles,part.assigned_angle)
        for i,record in enumerate(normal_records):
            prior=input_records[(scenario,seed,record['relative_path'])]
            assert prior['angle']==int(angles[i]) and prior['source_sha256']==record['file_sha256']
            assert array_hash(inputs[i])==prior['preprocessed_sha256']
    if scenario=='unrotated':
        assert all(array_hash(inputs[i])==r['preprocessed_sha256'] for i,r in enumerate(normal_records))
    rel=f'03_new_model_evaluation/{scenario}_seed{seed}.npz'
    p=register_source(SOURCE_RUN/rel,source_output_hashes[rel])
    with np.load(p,allow_pickle=False) as data:
        assert data['relative_paths'].tolist()==all_paths
        assert np.array_equal(data['angles'][normal_indices],angles)
        reconstructed={k:data[f'{k}_reconstruction'][normal_indices].copy() for k in CONDITIONS}
    assert inputs.dtype==np.float32 and inputs.shape==(len(normal_indices),128,128)
    assert np.isfinite(inputs).all() and inputs.min()>=0 and inputs.max()<=1
    assert all(v.shape==inputs.shape and np.isfinite(v).all() and v.min()>=0 and v.max()<=1 for v in reconstructed.values())
    case_data[(scenario,seed)]={'inputs':inputs,'angles':angles,'reconstructions':reconstructed}
print(f'正常{len(normal_paths)}枚 × {len(CASES)}ケース × 3モデルを比較します。')
display(pd.DataFrame([{'検査条件':SCENARIOS[s],'割り当てseed':seed,'正常枚数':len(normal_paths)} for s,seed in CASES]))


## 2. 回転参照画像と、角度を推定する計算

元の1024画像を1度刻みで回転し、既存と同じ反射補完・最近傍回転・元領域画素維持・LANCZOS128・白黒化を行います。
参照画像は学習画像ではなく、**同じ正常テスト画像を回転した照合用画像**です。

参照と対象を同じようにぼかし、領域を切り出し、64×64へそろえます。
各画像の平均を引いて長さを1にしたベクトルの内積を計算します。

$$c(\theta)=\frac{\sum_p (q_p-\bar q)(t_{\theta,p}-\bar t_\theta)}
 {\sqrt{\sum_p(q_p-\bar q)^2}\sqrt{\sum_p(t_{\theta,p}-\bar t_\theta)^2}},\qquad
 \hat\theta=\mathop{\mathrm{argmax}}_{\theta\in\{0,\ldots,359\}}c(\theta)$$

ここで$q$は対象画像、$t_\theta$は角度$\theta$の参照画像。平行移動・倍率調整は行いません。
平均輝度・全体コントラストの差を調整しますが、ぼけ・変形・繰り返し模様の影響は残ります。
中央領域にも背景は含まれます。「画像全体」と「中央」で角度が違う場合は要注意です。

角度差は $d(a,b)=|((a-b+180)\bmod360)-180|$ とします（359度と0度は1度差）。


In [ ]:
def rotation_pad_width(size):
    return math.ceil((math.hypot(*size)-min(size))/2)+4

def rotate_with_reflect(image,angle,verify_support=False):
    if angle==0: return image.copy()
    width,height=image.size
    pad=rotation_pad_width(image.size)
    pixels=np.asarray(image)
    extended=Image.fromarray(np.pad(pixels,((pad,pad),(pad,pad),(0,0)),mode='reflect'))
    box=(pad,pad,pad+width,pad+height)
    options=dict(resample=Image.Resampling.NEAREST,expand=False,center=None)
    out=extended.rotate(angle,fillcolor=(255,255,255),**options).crop(box)
    mask=Image.new('L',image.size,255).rotate(angle,fillcolor=0,**options)
    original_part=image.rotate(angle,fillcolor=(255,255,255),**options)
    valid=np.asarray(mask)>0
    result=np.asarray(out).copy()
    result[valid]=np.asarray(original_part)[valid]
    if verify_support:
        support=Image.new('L',extended.size,255).rotate(angle,fillcolor=0,**options).crop(box)
        assert np.all(np.asarray(support)==255),'パディングが不足しています。'
        assert np.array_equal(result[valid],np.asarray(original_part)[valid])
    return Image.fromarray(result)


def reference_array(image,angle):
    rotated=rotate_with_reflect(image,float(angle),verify_support=True)
    return np.array(rotated.resize((128,128),resample=Image.Resampling.LANCZOS).convert('L'),dtype=np.float32)/np.float32(255)

def circular_signed(a,b):return (np.asarray(a,dtype=float)-np.asarray(b,dtype=float)+180.0)%360.0-180.0
def circular_distance(a,b):return np.abs(circular_signed(a,b))
def nearest_training_angles(angle):
    distances=circular_distance(TRAINING_ANGLES,angle)
    return TRAINING_ANGLES[np.isclose(distances,distances.min(),rtol=0,atol=1e-9)]

def blur_array(array,sigma):
    # 推定用画像だけuint8化してPillowのぼかしを使用。保存済み入力・復元は変更しない。
    if sigma==0:return np.asarray(array,dtype=np.float32).copy()
    image=Image.fromarray(np.rint(np.clip(array,0,1)*255).astype(np.uint8))
    return np.array(image.filter(ImageFilter.GaussianBlur(radius=float(sigma))),dtype=np.float32)/np.float32(255)

def image_feature(array,roi):
    image=blur_array(array,ANALYSIS_BLUR_SIGMA)
    left,top,right,bottom=ROI_BOXES[roi]
    assert 0<=left<right<=128 and 0<=top<bottom<=128
    part=image[top:bottom,left:right]
    # Fモードでfloat32のまま縮小し、特徴作成時の追加のuint8量子化を避ける。
    resized=np.asarray(Image.fromarray(part).resize((FEATURE_SIZE,FEATURE_SIZE),resample=Image.Resampling.LANCZOS),dtype=np.float32)
    vector=resized.ravel().copy();vector-=vector.mean()
    norm=float(np.linalg.norm(vector))
    if not np.isfinite(norm) or norm<=1e-8:return np.zeros_like(vector),False
    return vector/norm,True

def feature_matrix(images,roi):
    values=[image_feature(image,roi) for image in images]
    return np.stack([v for v,valid in values]),np.array([valid for v,valid in values])

def estimate_one(correlations,valid_query,valid_templates):
    values=np.asarray(correlations,dtype=float).copy()
    values[~valid_templates]=-np.inf
    if not valid_query or not np.isfinite(values).any():
        return {'estimable':False,'estimated_angle':np.nan,'best_correlation':np.nan,
            'second_angle':np.nan,'peak_gap':np.nan,'correlation_at_opposite':np.nan,
            'exact_tie_count':0,'ambiguous':True}
    best_index=int(np.argmax(values));best_angle=int(CANDIDATE_ANGLES[best_index])
    outside=circular_distance(CANDIDATE_ANGLES,best_angle)>OTHER_PEAK_EXCLUSION
    candidate=np.where(outside,values,-np.inf);second_index=int(np.argmax(candidate))
    has_second=bool(np.isfinite(candidate[second_index]))
    gap=float(values[best_index]-candidate[second_index]) if has_second else np.nan
    tie_count=int(np.isclose(values,values[best_index],rtol=0,atol=1e-6).sum())
    return {'estimable':True,'estimated_angle':best_angle,'best_correlation':float(values[best_index]),
        'second_angle':int(CANDIDATE_ANGLES[second_index]) if has_second else np.nan,'peak_gap':gap,
        'correlation_at_opposite':float(values[(best_index+180)%360]),'exact_tie_count':tie_count,
        'ambiguous':bool(not has_second or gap<SMALL_PEAK_GAP or values[best_index]<LOW_CORRELATION or tie_count>1)}

def angle_measures(estimate,input_angle):
    nearest=nearest_training_angles(input_angle)
    base={'input_angle':float(input_angle),'nearest_training_angles':','.join(map(str,nearest)),
        'input_distance_to_grid':float(circular_distance(nearest,input_angle).min())}
    if not estimate['estimable']:
        return dict(base,input_angle_error=np.nan,distance_to_any_training_angle=np.nan,
            distance_to_nearest_input_training_angle=np.nan,diagnostic='推定不可')
    angle=estimate['estimated_angle']
    error=float(circular_distance(angle,input_angle))
    nearest_error=float(circular_distance(nearest,angle).min())
    grid_error=float(circular_distance(TRAINING_ANGLES,angle).min())
    # 入力自体が学習角度に近い例では、「追従」と「学習角度への偏り」を区別しにくい。
    informative=base['input_distance_to_grid']>2*ANGLE_TOLERANCE
    if error<=ANGLE_TOLERANCE:diagnostic='入力角度に近い'
    elif not informative:diagnostic='入力が学習角度に近く区別困難'
    elif nearest_error<=ANGLE_TOLERANCE:diagnostic='最寄り学習角度への偏り候補'
    elif grid_error<=ANGLE_TOLERANCE:diagnostic='別の学習角度に近い'
    else:diagnostic='入力・学習角度のいずれにも近くない'
    return dict(base,input_angle_error=error,distance_to_any_training_angle=grid_error,
        distance_to_nearest_input_training_angle=nearest_error,diagnostic=diagnostic)


## 3. 推定と校正をまとめて実行する（モデル計算なし）

元画像ごとに360参照を作り、その画像の全ケース・全モデルを照合します。
全60枚で21,600枚の高解像度回転参照を作るため、処理時間がかかります。元画像ごとに処理し、参照画像全体は保存しません。
相関曲線は保存し、可視化に再学習・再推論は不要です。

校正は、角度が既知の保存済み入力に追加のぼかし0・1・2画素を与えて行います。
モデルの復元を擬似的に作るものではなく、**ぼけによる角度推定の誤差**を調べる補助です。
形状の崩れなどを含む実際の復元画像で、同じ推定精度が保証されるわけではありません。


In [ ]:
RUN_DIR=OUTPUT_BASE/new_id();RUN_DIR.mkdir(parents=True,exist_ok=False)
CURVE_DIR=RUN_DIR/'correlation_curves';CURVE_DIR.mkdir()
FIGURE_DIR=RUN_DIR/'figures';FIGURE_DIR.mkdir()
config={'status':'running','created_at_jst':datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
    'source_run':SOURCE_RUN.relative_to(ROOT).as_posix(),'source_model_hashes':evaluation_config['model_hashes'],
    'normal_paths':normal_paths,'case_count':len(CASES),'conditions':list(CONDITIONS),
    'candidate_angles':CANDIDATE_ANGLES.tolist(),'training_angles':TRAINING_ANGLES.tolist(),
    'roi_boxes':ROI_BOXES,'feature_size':FEATURE_SIZE,'analysis_blur_sigma':ANALYSIS_BLUR_SIGMA,
    'calibration_blurs':list(CALIBRATION_BLURS),'angle_tolerance':ANGLE_TOLERANCE,
    'other_peak_exclusion':OTHER_PEAK_EXCLUSION,'small_peak_gap':SMALL_PEAK_GAP,
    'low_correlation':LOW_CORRELATION,'source_hashes':source_hashes,
    'versions':{k:importlib.metadata.version(k) for k in ['numpy','pandas','Pillow','matplotlib']},
    'python':platform.python_version(),'retained_curves':RETAIN_CORRELATION_CURVES,
    'estimation':'same-image rotation template, zero-mean unit-norm correlation; no translation/scale registration'}
write_json(RUN_DIR/'config.json',config)
write_json(RUN_DIR/'normal_manifest.json',normal_records)
write_json(RUN_DIR/'case_definitions.json',[{'scenario':s,'assignment_seed':seed} for s,seed in CASES])
result_rows=[];calibration_rows=[]
try:
    for index,path in enumerate(normal_paths):
        templates=np.stack([reference_array(originals[path],angle) for angle in CANDIDATE_ANGLES])
        # 既知の整数角度で作った参照が、保存済み入力そのものと一致することを先に確認する。
        for case in CASES:
            true_angle=float(case_data[case]['angles'][index])
            assert true_angle.is_integer()
            assert np.array_equal(templates[int(true_angle)],case_data[case]['inputs'][index]),f'参照前処理不一致：{path} {case}'
        query_metadata=[];queries=[];cal_metadata=[];cal_queries=[]
        for scenario,seed in CASES:
            stored=case_data[(scenario,seed)];angle=float(stored['angles'][index])
            for condition in CONDITIONS:
                queries.append(stored['reconstructions'][condition][index])
                query_metadata.append({'relative_path':path,'scenario':scenario,'assignment_seed':seed,
                    'condition':condition,'input_angle':angle})
            for sigma in CALIBRATION_BLURS:
                cal_queries.append(blur_array(stored['inputs'][index],sigma))
                cal_metadata.append({'relative_path':path,'scenario':scenario,'assignment_seed':seed,
                    'additional_blur_sigma':sigma,'input_angle':angle})
        for roi in ROI_BOXES:
            matrix,template_valid=feature_matrix(templates,roi)
            features,query_valid=feature_matrix(queries,roi)
            cal_features,cal_valid=feature_matrix(cal_queries,roi)
            correlations=np.clip(features@matrix.T,-1.0,1.0)
            cal_correlations=np.clip(cal_features@matrix.T,-1.0,1.0)
            for j,meta in enumerate(query_metadata):
                estimate=estimate_one(correlations[j],query_valid[j],template_valid)
                result_rows.append(dict(meta,roi=roi,**estimate,**{k:v for k,v in angle_measures(estimate,meta['input_angle']).items() if k!='input_angle'}))
            for j,meta in enumerate(cal_metadata):
                estimate=estimate_one(cal_correlations[j],cal_valid[j],template_valid)
                calibration_rows.append(dict(meta,roi=roi,**estimate,**{k:v for k,v in angle_measures(estimate,meta['input_angle']).items() if k!='input_angle'}))
            if RETAIN_CORRELATION_CURVES:
                np.savez_compressed(CURVE_DIR/f'{Path(path).stem}_{roi}.npz',candidate_angles=CANDIDATE_ANGLES,
                    correlations=correlations,template_valid=template_valid,query_valid=query_valid,
                    calibration_correlations=cal_correlations,calibration_valid=cal_valid)
        if RETAIN_CORRELATION_CURVES:
            write_json(CURVE_DIR/f'{Path(path).stem}_query_order.json',
                {'relative_path':path,'reconstruction_queries':query_metadata,'calibration_queries':cal_metadata})
        del templates,matrix,features,cal_features,correlations,cal_correlations
        if (index+1)%5==0 or index+1==len(normal_paths):
            pd.DataFrame(result_rows).to_csv(RUN_DIR/'angle_estimates.csv',index=False)
            pd.DataFrame(calibration_rows).to_csv(RUN_DIR/'calibration_estimates.csv',index=False)
            print(f'角度推定：{index+1}/{len(normal_paths)}画像',flush=True)
    results=pd.DataFrame(result_rows);calibration=pd.DataFrame(calibration_rows)
    assert len(results)==len(normal_paths)*len(CASES)*len(CONDITIONS)*len(ROI_BOXES)
    assert not results.duplicated(['relative_path','scenario','assignment_seed','condition','roi']).any()
    config.update(status='estimated');write_json(RUN_DIR/'config.json',config)
except BaseException as exc:
    pd.DataFrame(result_rows).to_csv(RUN_DIR/'angle_estimates.csv',index=False)
    pd.DataFrame(calibration_rows).to_csv(RUN_DIR/'calibration_estimates.csv',index=False)
    config.update(status='failed_or_interrupted',error=f'{type(exc).__name__}: {exc}')
    write_json(RUN_DIR/'config.json',config);raise
print('保存先：',RUN_DIR)


## 4. 最初に、角度推定方法の校正結果を確認する

追加のぼかしの強さごとに、角度誤差・3度以内の割合・推定不可・曖昧さを確認します。
校正で誤りが多い領域・角度の復元推定は、慎重に扱ってください。特に180度違いの候補にも注意します。
相関差0.02・相関0.30は確認用の目安で、正しい角度である確率を表しません。
以下の集計は全例を含めます。推定が曖昧な例を黙って除外しません。


In [ ]:
def summarize_angles(group):
    valid=group[group.estimable]
    errors=valid.input_angle_error
    return pd.Series({'sample_count':len(group),'estimable_count':len(valid),
        'unestimable_count':int((~group.estimable).sum()),'ambiguous_count':int(group.ambiguous.sum()),
        'mean_error':errors.mean(),'median_error':errors.median(),'p90_error':errors.quantile(.90),
        'within_tolerance_fraction_all':float((group.input_angle_error<=ANGLE_TOLERANCE).sum()/len(group)),
        'near_180_error_fraction_all':float((group.input_angle_error>=180-ANGLE_TOLERANCE).sum()/len(group))})
cal_summary=[]
for keys,part in calibration.groupby(['roi','additional_blur_sigma','scenario'],sort=False):
    cal_summary.append(dict(zip(['roi','additional_blur_sigma','scenario'],keys),**summarize_angles(part).to_dict()))
cal_summary=pd.DataFrame(cal_summary);cal_summary.to_csv(RUN_DIR/'calibration_summary.csv',index=False)
display(cal_summary.replace({'roi':ROI_LABELS,'scenario':SCENARIOS}))
fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
for ax,roi in zip(axes,ROI_BOXES):
    groups=[calibration[(calibration.roi==roi)&(calibration.additional_blur_sigma==s)].input_angle_error.dropna().to_numpy() for s in CALIBRATION_BLURS]
    if all(len(g) for g in groups):ax.boxplot(groups,tick_labels=[str(s) for s in CALIBRATION_BLURS],showfliers=True)
    ax.axhline(ANGLE_TOLERANCE,color='gray',ls='--',label=f'目安：{ANGLE_TOLERANCE:g}度')
    ax.set(title=ROI_LABELS[roi]+'：ぼかした入力の推定誤差',xlabel='追加ぼかしのσ（128画像の画素単位）',ylabel='正しい入力角度との円周距離（度）')
    ax.grid(axis='y',ls='--',alpha=.4);ax.legend()
fig.savefig(FIGURE_DIR/'calibration_errors.png',dpi=180);plt.show();plt.close(fig)
print('誤差の大きい校正例：')
display(calibration.sort_values('input_angle_error',ascending=False).head(20))


## 5. 入力への追従と、学習角度への偏りを数値で比較する

- **入力角度誤差**：小さいほど入力の向きを保っている。
- **学習角度への距離**：36度刻みのどれかに近いか。これだけでは最寄り角度を選んだ証拠にならない。
- **入力に最寄りの学習角度への距離**：最寄り候補に近いか。中間角度では両候補を認める。
- 入力が学習角度から6度より離れた例に限り、入力誤差が3度超・最寄り候補まで3度以内なら「偏り候補」とする。

この分類は説明用の操作的定義です。結果は3度の設定に依存するため、連続量・相関曲線・校正と一緒に読みます。
通常モデルは0度しか学習していません。3モデルへ同じ36度グリッドを重ねるのは比較基準であり、
全モデルがその10角度を学習したという意味ではありません。

5回のケースは各回の正常画像で集計した後、平均と標準偏差を出します。
繰り返し画像を独立標本とする有意差検定や、モデルの機構を断定する判定は行いません。


In [ ]:
trial_rows=[]
for keys,part in results.groupby(['roi','scenario','assignment_seed','condition'],sort=False):
    row=dict(zip(['roi','scenario','assignment_seed','condition'],keys),**summarize_angles(part).to_dict())
    informative=part[part.input_distance_to_grid>2*ANGLE_TOLERANCE]
    row.update(mean_distance_to_grid=part.distance_to_any_training_angle.mean(),
        mean_distance_to_nearest=part.distance_to_nearest_input_training_angle.mean(),
        informative_count=len(informative),
        snapping_candidate_fraction=(float((informative.diagnostic=='最寄り学習角度への偏り候補').sum()/len(informative)) if len(informative) else np.nan))
    trial_rows.append(row)
trials=pd.DataFrame(trial_rows);trials.to_csv(RUN_DIR/'trial_summary.csv',index=False)
summary=trials.groupby(['roi','scenario','condition'],sort=False).agg(
    assignment_count=('assignment_seed','size'),mean_error=('mean_error','mean'),std_error_across_assignments=('mean_error','std'),
    mean_distance_to_grid=('mean_distance_to_grid','mean'),mean_distance_to_nearest=('mean_distance_to_nearest','mean'),
    input_following_fraction=('within_tolerance_fraction_all','mean'),
    snapping_candidate_fraction=('snapping_candidate_fraction','mean'),
    ambiguous_count_mean=('ambiguous_count','mean'),unestimable_count_mean=('unestimable_count','mean')).reset_index()
summary.to_csv(RUN_DIR/'angle_summary.csv',index=False)
display(summary.replace({'roi':ROI_LABELS,'scenario':SCENARIOS,'condition':LABELS}))
diagnostic_counts=results.groupby(['roi','scenario','assignment_seed','condition','diagnostic'],sort=False).size().rename('count').reset_index()
diagnostic_counts.to_csv(RUN_DIR/'diagnostic_counts.csv',index=False)
# ROI間の不一致は、背景と中央の異なる復元、または推定の不安定さの手がかり。
roi_comparison=results.pivot(index=['relative_path','scenario','assignment_seed','condition'],columns='roi',values='estimated_angle').reset_index()
roi_comparison['roi_angle_disagreement']=circular_distance(roi_comparison['whole'],roi_comparison['central'])
roi_comparison.to_csv(RUN_DIR/'roi_comparison.csv',index=False)
display(roi_comparison.sort_values('roi_angle_disagreement',ascending=False).head(20))


In [ ]:
for scenario in SCENARIOS:
    part=results[results.scenario==scenario]
    fig,axes=plt.subplots(2,3,figsize=(16,10),constrained_layout=True)
    for row,roi in enumerate(ROI_BOXES):
        for col,condition in enumerate(CONDITIONS):
            selected=part[(part.roi==roi)&(part.condition==condition)&part.estimable]
            y=selected.input_angle+circular_signed(selected.estimated_angle,selected.input_angle)
            ax=axes[row,col]
            ax.scatter(selected.input_angle,y,s=15,alpha=.35,color=COLORS[condition],label='推定復元角度')
            ax.plot([0,360],[0,360],ls='--',color='black',lw=1,label='入力へ追従')
            for angle in np.arange(-180,541,36):ax.axhline(angle,color='gray',lw=.5,alpha=.25)
            ax.set(xlim=(-5,365),ylim=(-180,540),xlabel='入力角度（度）',ylabel='推定復元角度（円周をほどいた表示）',title=f'{LABELS[condition]} / {ROI_LABELS[roi]}')
            ax.legend(fontsize=9);ax.grid(ls='--',alpha=.2)
    fig.suptitle(SCENARIOS[scenario]+'：対角線なら追従、横線付近への集中はグリッドへの偏り候補')
    fig.savefig(FIGURE_DIR/f'estimated_vs_input_{scenario}.png',dpi=180);plt.show();plt.close(fig)

for roi in ROI_BOXES:
    fig,axes=plt.subplots(2,3,figsize=(16,9),constrained_layout=True)
    for row,scenario in enumerate(['midpoint','unseen_random']):
        for col,(key,title,bins) in enumerate([
            ('input_angle_error','入力角度との誤差',np.linspace(0,180,37)),
            ('distance_to_any_training_angle','36度グリッドへの距離',np.linspace(0,18,19)),
            ('distance_to_nearest_input_training_angle','入力に最寄りの学習角度への距離',np.linspace(0,180,37))]):
            ax=axes[row,col]
            for condition in CONDITIONS:
                values=results[(results.roi==roi)&(results.scenario==scenario)&(results.condition==condition)][key].dropna()
                if len(values):ax.hist(values,bins=bins,weights=np.ones(len(values))/len(values),histtype='step',lw=2,color=COLORS[condition],label=LABELS[condition])
            ax.set(title=SCENARIOS[scenario]+' / '+title,xlabel='円周距離（度）',ylabel='推定可能例内の割合')
            ax.legend();ax.grid(ls='--',alpha=.3)
    fig.suptitle(ROI_LABELS[roi]+'：正常画像は各5回で繰り返している（独立300枚ではない）')
    fig.savefig(FIGURE_DIR/f'angle_error_distributions_{roi}.png',dpi=180);plt.show();plt.close(fig)


## 6. 代表画像と相関曲線で、推定結果を確かめる

入力／実際の復元／推定角度の参照／入力に最寄りの学習角度の参照を並べます。
中間角度では最寄り候補が2つあるので両方表示します。推定が最寄りと別の角度なら、その差も確認してください。
画像は見やすさのため128のまま表示し、角度推定は設定したぼかし・領域・特徴解像度で行っています。
相関曲線に複数の高い山がある場合は、1つの角度を確定値として扱わないでください。

`VISUAL_PATHS`・`VISUAL_CASES`・`VISUAL_ROIS` は、設定セルを再実行せずに下で変更してから表示セルのみ再実行できます。
これにより、推定処理をやり直さずに曖昧な例を調べられます。ケースは上で評価済みのものを選んでください。


In [ ]:
# 変更例：VISUAL_PATHS=['good/001.png']; VISUAL_CASES=[('midpoint',43)]
paths_to_show=normal_paths[:1] if VISUAL_PATHS is None else VISUAL_PATHS
assert set(paths_to_show).issubset(set(normal_paths))
assert set(VISUAL_CASES).issubset(set(CASES))
assert set(VISUAL_ROIS).issubset(set(ROI_BOXES))
for path in paths_to_show:
    index=normal_paths.index(path)
    for scenario,seed in VISUAL_CASES:
        stored=case_data[(scenario,seed)];angle=float(stored['angles'][index]);nearest=nearest_training_angles(angle)
        references={int(a):reference_array(originals[path],a) for a in nearest}
        for roi in VISUAL_ROIS:
            fig,axes=plt.subplots(3,3+len(nearest),figsize=(3.1*(3+len(nearest)),10),squeeze=False,constrained_layout=True)
            selected=results[(results.relative_path==path)&(results.scenario==scenario)&(results.assignment_seed==seed)&(results.roi==roi)].set_index('condition')
            for row,condition in enumerate(CONDITIONS):
                estimate=selected.loc[condition]
                axes[row,0].imshow(stored['inputs'][index],cmap='gray',vmin=0,vmax=1)
                axes[row,0].set_title(f'入力 {angle:g}度')
                axes[row,1].imshow(stored['reconstructions'][condition][index],cmap='gray',vmin=0,vmax=1)
                axes[row,1].set_title('保存済みの復元')
                if estimate.estimable:
                    a=int(estimate.estimated_angle)
                    axes[row,2].imshow(reference_array(originals[path],a),cmap='gray',vmin=0,vmax=1)
                    axes[row,2].set_title(f'推定 {a}度 / 誤差{estimate.input_angle_error:g}度')
                else:axes[row,2].text(.5,.5,'推定不可',ha='center',va='center')
                for col,a in enumerate(nearest,3):
                    axes[row,col].imshow(references[int(a)],cmap='gray',vmin=0,vmax=1)
                    axes[row,col].set_title(f'入力に最寄り {a}度')
                axes[row,0].set_ylabel(LABELS[condition])
                for col in [0,1]:
                    left,top,right,bottom=ROI_BOXES[roi]
                    axes[row,col].add_patch(plt.Rectangle((left-.5,top-.5),right-left,bottom-top,fill=False,edgecolor='cyan',lw=1))
                for ax in axes[row]:ax.set_xticks([]);ax.set_yticks([])
            fig.suptitle(f'{path} / {SCENARIOS[scenario]} / seed{seed} / 推定領域：{ROI_LABELS[roi]}')
            stem=f'{Path(path).stem}_{scenario}_{seed}_{roi}'
            fig.savefig(FIGURE_DIR/f'example_{stem}.png',dpi=150);plt.show();plt.close(fig)
            display(selected[['input_angle','estimated_angle','input_angle_error','distance_to_nearest_input_training_angle','best_correlation','second_angle','peak_gap','ambiguous','diagnostic']].rename(index=LABELS))
            if RETAIN_CORRELATION_CURVES:
                order=read_json(CURVE_DIR/f'{Path(path).stem}_query_order.json')['reconstruction_queries']
                with np.load(CURVE_DIR/f'{Path(path).stem}_{roi}.npz',allow_pickle=False) as saved:
                    curves=saved['correlations'].copy()
                fig,axes=plt.subplots(3,1,figsize=(12,9),constrained_layout=True)
                for ax,condition in zip(axes,CONDITIONS):
                    j=next(i for i,r in enumerate(order) if r['scenario']==scenario and r['assignment_seed']==seed and r['condition']==condition)
                    ax.plot(CANDIDATE_ANGLES,curves[j],color=COLORS[condition],label=LABELS[condition])
                    for a in TRAINING_ANGLES:ax.axvline(a,color='gray',lw=.6,alpha=.4)
                    ax.axvline(angle,color='black',ls='--',label='入力角度')
                    estimate=selected.loc[condition]
                    if estimate.estimable:ax.axvline(estimate.estimated_angle,color='crimson',ls=':',label='推定角度')
                    ax.set(xlim=(0,359),ylim=(-1.05,1.05),ylabel='正規化相関',xlabel='参照角度（度）');ax.legend(loc='lower right');ax.grid(ls='--',alpha=.3)
                fig.suptitle(f'{path} / {angle:g}度入力 / {ROI_LABELS[roi]}：灰色は36度グリッド')
                fig.savefig(FIGURE_DIR/f'correlation_{stem}.png',dpi=180);plt.show();plt.close(fig)


## 7. 結果の読み取りと限界

- まずぼかした入力の校正を確認する。校正が良くても、変形した復元画像の角度が正確とは限らない。
- Bの推定角度が36度グリッドに近いだけでは不十分。**入力に最寄りの候補**との対応、A/Cとの差、ROIの一致、相関曲線も確認する。
- 入力から離れた別の学習角度に似る場合は、「最寄り角度を選ぶ」という仮説とは異なる。
- 背景の線・黒丸・本体・端子が別々の向きになる復元では、画像全体に単一角度を付けること自体が難しい。
- 小さな相関差、180度の類似、低相関、複数の山、領域間の不一致は、角度推定が不安定な手がかり。
- 本検証は保存済みの正常画像の形・向きを調べる補助分析で、異常検知のAUROC評価を置き換えない。
- 一致度を見てモデル・学習条件を選ぶ検証ではなく、既存モデルの探索的な診断。学習seedは1つで、同じ正常画像を繰り返している。
- 角度の分解能、ぼかし・領域・許容幅に依存するため、偏りの候補を観察しても内部での角度選択を証明したとは言わない。
- transistorの背景や向きに関する結果を、そのままbottleへ一般化しない。

再表示には、今回のCSV・相関曲線と参照元の保存配列を使います。
別セッションで再表示する場合は共通の設定・関数・参照読込を実行し、推定セルの代わりに
`RUN_DIR`を保存先に指定、`results=pd.read_csv(RUN_DIR/'angle_estimates.csv')` 等で読み込んでください。
設定や参照データが同じであることを`config.json`で確認します。


In [ ]:
assert all(sha256(ROOT/path)==value for path,value in source_hashes.items()),'参照元が変更されています。'
config.update(status='completed',completed_at_jst=datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
    estimate_count=len(results),calibration_count=len(calibration),normal_source_count=len(normal_paths))
write_json(RUN_DIR/'config.json',config)
write_json(RUN_DIR/'output_file_hashes.json',{p.relative_to(RUN_DIR).as_posix():sha256(p)
    for p in sorted(RUN_DIR.rglob('*')) if p.is_file() and p.name!='output_file_hashes.json'})
print('保存先：',RUN_DIR)
print('既存モデル・入力・復元ファイルは変更していません。新規学習・モデル推論は行っていません。')
